# BirdCLEF 2026 — Pseudo-Label Inference (Timing Run)

Runs the ensemble (ConvNeXt-Small v5 + ECA-NFNet-L0 v9) on a small sample of unlabeled
train soundscapes to measure per-step inference time and extrapolate the full runtime.

In [ ]:
! pip install -q fastai timm

In [ ]:
import time
import json
import warnings
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torchvision import transforms as T
from fastai.vision.all import load_learner

warnings.filterwarnings('ignore', category=UserWarning, module='fastai')

In [ ]:
import kagglehub
competition_dir = Path(kagglehub.competition_download('birdclef-2026'))
print('Competition dir:', competition_dir)

labels_df = pd.read_csv(competition_dir / 'train_soundscapes_labels.csv')
labeled_files = set(labels_df['filename'].unique())
print(f'Labeled soundscape files (excluded from pseudo-labeling): {len(labeled_files)}')

all_soundscapes = sorted((competition_dir / 'train_soundscapes').glob('*.ogg'))
unlabeled = [f for f in all_soundscapes if f.name not in labeled_files]
print(f'Total train soundscapes: {len(all_soundscapes)}')
print(f'Unlabeled (pseudo-label targets): {len(unlabeled)}')

In [ ]:
MODEL_A_PATH = '/kaggle/input/models/ucheozoemena/bird-clef-classifier/pytorch/multilabel_234/5/model_multilabel_234.pkl'
MODEL_B_PATH = '/kaggle/input/models/ucheozoemena/bird-clef-classifier/pytorch/multilabel_234/9/model_multilabel_234.pkl'

HOP_LENGTH     = 512
SAMPLE_RATE    = 32000
CLIP_DURATION  = 5
STRIDE_DURATION = 2.5
TARGET_SIZE    = (224, 224)
BATCH_SIZE     = 64
N_SAMPLE       = 20   # files to time; increase for a longer calibration run

In [ ]:
t0 = time.perf_counter()
learn_a = load_learner(MODEL_A_PATH)
t1 = time.perf_counter()
learn_b = load_learner(MODEL_B_PATH)
t2 = time.perf_counter()
print(f'Model A (ConvNeXt-Small) loaded in {t1-t0:.1f}s')
print(f'Model B (ECA-NFNet-L0)   loaded in {t2-t1:.1f}s')

learn_a.model.eval()
learn_b.model.eval()
device = next(learn_a.model.parameters()).device
print(f'Running on: {device}')

In [ ]:
clip_length    = int(CLIP_DURATION * SAMPLE_RATE)
stride_samples = int(STRIDE_DURATION * SAMPLE_RATE)

tfm = T.Compose([
    T.Resize(TARGET_SIZE),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def window_to_img(window):
    lo, hi = float(window.min()), float(window.max())
    arr = np.zeros_like(window, dtype=np.uint8) if hi == lo else (
        (window - lo) / (hi - lo) * 255
    ).astype(np.uint8)
    return Image.fromarray(arr).resize(TARGET_SIZE).convert('RGB')

def build_clip_batch(S_db, hop_length, n_samples):
    frames_per_clip = int(CLIP_DURATION * SAMPLE_RATE / hop_length)
    stride_frames   = int(STRIDE_DURATION * SAMPLE_RATE / hop_length)
    imgs = []
    k = 0
    while k * stride_samples + clip_length <= n_samples:
        window = S_db[:, k * stride_frames : k * stride_frames + frames_per_clip]
        imgs.append(window_to_img(window))
        k += 1
    return imgs

def run_inference(learn, imgs):
    preds = []
    for i in range(0, len(imgs), BATCH_SIZE):
        batch = torch.stack([tfm(img) for img in imgs[i:i+BATCH_SIZE]]).to(device)
        with torch.no_grad():
            logits = learn.model(batch)
        preds.append(torch.sigmoid(logits).cpu().numpy())
    return np.vstack(preds)

In [ ]:
sample_files = unlabeled[:N_SAMPLE]

timings = []
wall_start = time.perf_counter()

for i, soundscape in enumerate(sample_files):
    row = {'file': soundscape.name}

    t = time.perf_counter()
    samples, _ = librosa.load(soundscape, sr=SAMPLE_RATE)
    row['t_load'] = time.perf_counter() - t

    t = time.perf_counter()
    S_db = librosa.power_to_db(
        librosa.feature.melspectrogram(y=samples, sr=SAMPLE_RATE, hop_length=HOP_LENGTH),
        ref=np.max,
    )
    row['t_mel'] = time.perf_counter() - t

    t = time.perf_counter()
    imgs = build_clip_batch(S_db, HOP_LENGTH, len(samples))
    row['n_clips'] = len(imgs)
    row['t_img_build'] = time.perf_counter() - t

    t = time.perf_counter()
    _ = run_inference(learn_a, imgs)
    row['t_infer_a'] = time.perf_counter() - t

    t = time.perf_counter()
    _ = run_inference(learn_b, imgs)
    row['t_infer_b'] = time.perf_counter() - t

    row['t_total'] = row['t_load'] + row['t_mel'] + row['t_img_build'] + row['t_infer_a'] + row['t_infer_b']
    timings.append(row)

    elapsed = time.perf_counter() - wall_start
    print(f'[{i+1:>2}/{N_SAMPLE}] {soundscape.name}  '
          f'load={row["t_load"]:.2f}s  mel={row["t_mel"]:.2f}s  '
          f'imgs={row["t_img_build"]:.2f}s  '
          f'infer_A={row["t_infer_a"]:.2f}s  infer_B={row["t_infer_b"]:.2f}s  '
          f'total={row["t_total"]:.2f}s  |  wall={elapsed:.0f}s elapsed')

print(f'\nTiming run complete: {N_SAMPLE} files in {time.perf_counter()-wall_start:.1f}s')

In [ ]:
df_t = pd.DataFrame(timings)
print('=== Per-step averages (seconds) ===')
for col in ['t_load', 't_mel', 't_img_build', 't_infer_a', 't_infer_b', 't_total']:
    print(f'  {col:<14}: mean={df_t[col].mean():.3f}s  min={df_t[col].min():.3f}s  max={df_t[col].max():.3f}s')

n_unlabeled = len(unlabeled)
mean_total  = df_t['t_total'].mean()
est_seconds = n_unlabeled * mean_total
est_hours   = est_seconds / 3600

print(f'\n=== Extrapolation ===')
print(f'Unlabeled files:       {n_unlabeled:,}')
print(f'Mean time per file:    {mean_total:.2f}s')
print(f'Estimated total time:  {est_seconds:,.0f}s  ({est_hours:.1f} hours)')
print(f'Kaggle GPU limit:      12 hours')
print(f'Fits in one run:       {"YES" if est_hours < 11 else "NO — split needed"}')

if est_hours >= 11:
    n_runs = int(np.ceil(est_hours / 10))
    files_per_run = int(np.ceil(n_unlabeled / n_runs))
    print(f'Suggested split:       {n_runs} runs of ~{files_per_run:,} files each')

df_t.to_csv('timing_results.csv', index=False)
print('\nTiming results saved to timing_results.csv')